In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
!pip install -q transformers datasets peft accelerate sentencepiece evaluate rouge-score huggingface_hub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [4]:
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
from peft import PeftModel
import evaluate

In [5]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ml_projects/NLP_Text_Summarization"
)
LORA_RESULTS_DIR = PROJECT_ROOT / "artifacts/lora_results"
LORA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using device:", device)

Using device: cpu


In [7]:
BASE_MODEL_NAME = "google/pegasus-large"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    use_safetensors=False,
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Error during conversion: ReadTimeout('The read operation timed out')


model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [8]:
LORA_MODEL_NAME = "hyperchancellor07/pegasus_lora"
model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL_NAME,
)
model.to(device)
print("LoRA adapters loaded successfully.")

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/12.6M [00:00<?, ?B/s]

LoRA adapters loaded successfully.


In [9]:
rouge = evaluate.load("rouge")

In [10]:
def generate_summary(dialogue_text):

    dialogue_text = "summarize: " + dialogue_text
    inputs = tokenizer(
        dialogue_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }
    summary_ids = model.generate(
        **inputs,
        max_length=64,
        min_length=8,
        num_beams=8,
        repetition_penalty=2.5,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
        early_stopping=True,
    )
    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
    )
    return generated_summary

In [11]:
raw_dataset = load_dataset("knkarthick/samsum")
sample_dialogue = raw_dataset["test"][0]["dialogue"]
reference_summary = raw_dataset["test"][0]["summary"]
generated_summary = generate_summary(sample_dialogue)
print("Dialogue:\n")
print(sample_dialogue)
print("Generated Summary:\n")
print(generated_summary)
print("Reference Summary:\n")
print(reference_summary)

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

Dialogue:

Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
Generated Summary:

Hannah" SUMMER. her It number will
Reference Summary:

Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [13]:
EVAL_SAMPLES = 50
generated_summaries = []
reference_summaries = []
for sample in raw_dataset["test"].select(range(EVAL_SAMPLES)):

    dialogue = sample["dialogue"]
    reference = sample["summary"]


    prediction = generate_summary(dialogue)


    generated_summaries.append(prediction)
    reference_summaries.append(reference)

KeyboardInterrupt: 

In [ ]:
results = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries,
)
results
results_df = pd.DataFrame([results])
results_path = (
    LORA_RESULTS_DIR / "lora_rouge_scores.csv"
)
results_df.to_csv(results_path, index=False)
print(f"Saved results to: {results_path}")

In [ ]:
predictions_df = pd.DataFrame({
    "reference_summary": reference_summaries,
    "generated_summary": generated_summaries,
})
prediction_output_path = (
    LORA_RESULTS_DIR / "lora_predictions.csv"
)
predictions_df.to_csv(
    prediction_output_path,
    index=False,
)
print(f"Saved predictions to: {prediction_output_path}")